Optical flow is the motion of brightness in a sequence of images estimated at per pixel velocity, describing how each pixel moves between frames.

In this paper, the first event-based version of optical flow is introduced due to event-based cameras high speed and low compute. 

Below is mini implementation of what the paper proposes.

In [8]:
from collections import deque
import numpy as np

# --- synthetic stream ---
event_stream = []
t0 = 0.0
dt_event = 10e-6
y = 200
x_start = 50
for i in range(20):
    event_stream.append((x_start + i, y, t0 + i * dt_event, +1))

H, W = 480, 640
alpha = 0.5
dt = 50e-6
n = 5
r = n // 2
lam = 1e-6  # small ridge to stabilize

Q = [[deque() for _ in range(W)] for _ in range(H)]

def trim_pixel_deque(y, x, cutoff):
    d = Q[y][x]
    while d and d[0][0] < cutoff:
        d.popleft()

def S(x, y, t1, t2):
    # bounds check
    if x < 0 or x >= W or y < 0 or y >= H:
        return 0.0
    # signed sum of polarities in [t1, t2)
    return float(sum(p for (tt, p) in Q[y][x] if t1 <= tt < t2))

for x0, y0, t, p in event_stream:
    # update buffer for this pixel
    Q[y0][x0].append((t, p))
    cutoff = t - dt

    # IMPORTANT: trim deques you will query in this local window
    for yy in range(max(0, y0 - r), min(H, y0 + r + 1)):
        for xx in range(max(0, x0 - r), min(W, x0 + r + 1)):
            trim_pixel_deque(yy, xx, cutoff)
            if xx - 1 >= 0:
                trim_pixel_deque(yy, xx - 1, cutoff)
            if yy - 1 >= 0:
                trim_pixel_deque(yy - 1, xx, cutoff)

    t1 = t - alpha * dt
    denom = (t - t1)
    if denom <= 0:
        continue

    A = np.zeros((n * n, 2), dtype=np.float32)
    b = np.zeros((n * n,), dtype=np.float32)

    idx = 0
    for yy in range(y0 - r, y0 + r + 1):
        for xx in range(x0 - r, x0 + r + 1):
            Ex = S(xx, yy, t - dt, t) - S(xx - 1, yy, t - dt, t)
            Ey = S(xx, yy, t - dt, t) - S(xx, yy - 1, t - dt, t)
            Et = S(xx, yy, t1, t) / denom

            A[idx, 0] = Ex
            A[idx, 1] = Ey
            b[idx] = Et
            idx += 1

    # Solve with ridge for stability: v = (A^T A + lam I)^-1 A^T b
    ATA = A.T @ A
    ATb = A.T @ b
    v = np.linalg.solve(ATA + lam * np.eye(2, dtype=np.float32), ATb)

    vx, vy = float(v[0]), float(v[1])
    print(f"t={t:.6f}  (x0,y0)=({x0},{y0})  vx={vx:.4f}  vy={vy:.4f}")


t=0.000000  (x0,y0)=(50,200)  vx=0.0000  vy=0.0000
t=0.000010  (x0,y0)=(51,200)  vx=13333.3291  vy=13333.3291
t=0.000020  (x0,y0)=(52,200)  vx=11428.5674  vy=17142.8535
t=0.000030  (x0,y0)=(53,200)  vx=0.0000  vy=19999.9961
t=0.000040  (x0,y0)=(54,200)  vx=0.0000  vy=19999.9961
t=0.000050  (x0,y0)=(55,200)  vx=0.0000  vy=19999.9961
t=0.000060  (x0,y0)=(56,200)  vx=0.0000  vy=19999.9961
t=0.000070  (x0,y0)=(57,200)  vx=0.0000  vy=19999.9961
t=0.000080  (x0,y0)=(58,200)  vx=0.0000  vy=19999.9961
t=0.000090  (x0,y0)=(59,200)  vx=0.0000  vy=19999.9961
t=0.000100  (x0,y0)=(60,200)  vx=0.0000  vy=19999.9961
t=0.000110  (x0,y0)=(61,200)  vx=0.0000  vy=19999.9961
t=0.000120  (x0,y0)=(62,200)  vx=0.0000  vy=19999.9961
t=0.000130  (x0,y0)=(63,200)  vx=0.0000  vy=19999.9961
t=0.000140  (x0,y0)=(64,200)  vx=0.0000  vy=19999.9961
t=0.000150  (x0,y0)=(65,200)  vx=0.0000  vy=19999.9961
t=0.000160  (x0,y0)=(66,200)  vx=0.0000  vy=19999.9961
t=0.000170  (x0,y0)=(67,200)  vx=0.0000  vy=19999.9961
t=0.00